# 📊 Yelp Review Summarizer — Evaluation Notebook

This notebook evaluates the LLM summarization quality using:
1. **Automated metrics**: format compliance, quote faithfulness, theme coverage, hallucination detection
2. **LLM-as-Judge**: Gemini 2.5 Flash scores summaries on faithfulness, completeness, coherence, relevance, and quote accuracy
3. **Model comparison**: Compare Qwen3-0.6B vs 1.7B vs 4B

**Prerequisites**: Run `colab_runner.ipynb` first to build the FAISS index.

---

## 1️⃣ Setup

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

import os
REPO_DIR = "/content/NLP-Final--Project"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/chucey/NLP-Final--Project.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/NLP-Final--Project'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 229 (delta 58), reused 88 (delta 40), pack-reused 112 (from 1)
Receiving objects: 100% (229/229), 41.43 MiB | 17.07 MiB/s, done.
Resolving deltas: 100% (107/107), done.
Working directory: /content/NLP-Final--Project


In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    python-dotenv \
    accelerate \
    google-generativeai gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
# Load FAISS index from Google Drive (must have been built previously)
DRIVE_INDEX_DIR = "/content/drive/MyDrive/NLP-Project/faiss_yelp"
LOCAL_INDEX_DIR = "faiss_yelp"

if os.path.exists(DRIVE_INDEX_DIR) and os.listdir(DRIVE_INDEX_DIR):
    !cp -r {DRIVE_INDEX_DIR} {LOCAL_INDEX_DIR}
    print("✅ FAISS index loaded from Drive")
elif not os.path.exists(LOCAL_INDEX_DIR):
    print("⚠️ No FAISS index found! Run colab_runner.ipynb first to build it.")
else:
    print("✅ FAISS index already exists locally")

⚠️ No FAISS index found! Run colab_runner.ipynb first to build it.


In [ ]:
# =============================================
# 🔑 Configure your Gemini API key here
# =============================================
GEMINI_API_KEY = ""
GEMINI_MODEL = "gemini-2.5-flash"
# =============================================

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA L4


## 2️⃣ Load Model & Vectorstore

In [6]:
import rag_retrival
from prompt import load_model, summarize_reviews
from zARCHIVE.evaluate import (
    auto_evaluate,
    llm_judge_evaluate,
    run_full_evaluation,
    evaluate_no_result_handling
)

print("Loading vectorstore...")
vs = rag_retrival.load_vectorstore()
print("✅ Vectorstore loaded!")

Loading vectorstore...


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open faiss_yelp/index.faiss for reading: No such file or directory

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
tok, model = load_model(MODEL_NAME)

Loading model: Qwen/Qwen3-0.6B ...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded on CUDA (dtype=torch.bfloat16)


## 3️⃣ Test: No-Result Handling

Verify that the model outputs "No reviews found" when no reviews are provided.

In [ ]:
print("Testing no-result handling...")
no_result_test = evaluate_no_result_handling(summarize_reviews, tok, model)
print(f"\n{'✅ ALL PASSED' if no_result_test['all_passed'] else '❌ SOME FAILED'}")

Testing no-result handling...
  ✅ Empty string: No reviews found for the given criteria.
  ✅ Whitespace only: No reviews found for the given criteria.
  ✅ Newlines only: No reviews found for the given criteria.

✅ ALL PASSED


## 4️⃣ Define Test Cases

In [ ]:
import time

# Define test cases with different filter combinations
TEST_CASES = [
    {
        "name": "Italian Restaurants (General)",
        "filters": {"categories": "Italian", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "Mexican Restaurants",
        "filters": {"categories": "Mexican", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "Coffee Shops",
        "filters": {"categories": "Coffee", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "1-Star Reviews Only",
        "filters": {"review_stars": 1, "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "5-Star Reviews Only",
        "filters": {"review_stars": 5, "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "No Results (Fake Business)",
        "filters": {"business_name": "ZZZZZ_NONEXISTENT_BUSINESS_12345", "k": 20},
    },
    {
        "name": "phrase query 1",
        "filters": {"business_name": None, "k": 20,"baseline":"Italian restaurant with cozy atmosphere" },
    },
    {
        "name": "phrase query 2",
        "filters": {"business_name": None, "k": 20,"baseline": 'Poor service and long wait times'},
    },
    {
        "name": "phrase query 3",
        "filters": {"business_name": None, "k": 20,"baseline":"Good ethnic food in modest even shabby environment" },
    }
]

print(f"Defined {len(TEST_CASES)} test cases")
for tc in TEST_CASES:
    print(f"  • {tc['name']}: {tc['filters']}")

Defined 9 test cases
  • Italian Restaurants (General): {'categories': 'Italian', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • Mexican Restaurants: {'categories': 'Mexican', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • Coffee Shops: {'categories': 'Coffee', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • 1-Star Reviews Only: {'review_stars': 1, 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • 5-Star Reviews Only: {'review_stars': 5, 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • No Results (Fake Business): {'business_name': 'ZZZZZ_NONEXISTENT_BUSINESS_12345', 'k': 20}
  • phrase query 1: {'business_name': None, 'k': 20, 'baseline': 'Italian resta

## 5️⃣ Run Evaluation Across All Test Cases

In [ ]:
all_results = []
from gensim.models.doc2vec import Doc2Vec,\
    TaggedDocument
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
similars=[]
def similarEval(summary,docs):
  tagged_data = [TaggedDocument(words=word_tokenize(doc.lower()),
                              tags=[str(i)]) for i,
               doc in enumerate(docs)]


  themodel=Doc2Vec(vector_size=40,min_count=2,epochs=50)
  themodel.build_vocab(tagged_data)
  themodel.train(tagged_data,total_examples=themodel.corpus_count,epochs=themodel.epochs)


  document_vectors = [themodel.infer_vector(
    word_tokenize(doc.lower())) for doc in docs]


#generatedReview='hitler would have loved it here'
  generatedReview='i like the food and atmosphere is serene'

  theVector=[themodel.infer_vector(
    word_tokenize(summary.lower()))]
  a=theVector[0]
  b=document_vectors
  import numpy as np
  a2=np.sqrt(np.sum(np.square(a)))
  b2=np.sqrt(np.sum(np.square(b),axis=1))  #, axis=1

  productsN=a*b
  sums=np.sum(productsN,axis=1) #, axis=1
  productsD=a2*b2


  final=sums/productsD

  listFinal=final.tolist()
  huh=sorted(listFinal,reverse=True) #key=lambda x:x[0]
  print('array simiarity')
  print(listFinal)
  print(huh)
  if len(huh) < 5:
    return huh[0]
  else:
    return (huh[0]+huh[1]+huh[2]+huh[3]+huh[4])/5
for i, tc in enumerate(TEST_CASES):
    print(f"\n{'='*70}")
    print(f"TEST CASE {i+1}/{len(TEST_CASES)}: {tc['name']}")
    print(f"{'='*70}")

    # Retrieve reviews
    filters = tc["filters"]
    adict={"business_name": filters.get("business_name"),
                "city": filters.get("city"),
                "state": filters.get("state"),
                "review_stars": filters.get("review_stars"),
                "categories": filters.get("categories")}
    docs = rag_retrival.retrieve_reviews_for_summary(
        vs,
        adict,
        filters.get("baseline"),
        filters.get("k", 20),False
    )

    print(f"Retrieved {len(docs)} chars of review data")

    # Generate summary
    print("Generating summary...")
    start_time = time.time()
    summary = summarize_reviews(docs, tok, model)
    gen_time = time.time() - start_time
    print(f"Summary generated in {gen_time:.1f}s")
    print(f"\n--- Generated Summary ---\n{summary}\n--- End Summary ---\n")

    # Run evaluation
    eval_result = run_full_evaluation(
        summary, docs,
        api_key=GEMINI_API_KEY,
        gemini_model=GEMINI_MODEL
    )

    print('the similairty score')
    #print(similarEval(summary,docs))
    theScore=similarEval(summary,docs)
    print(theScore)
    print('got here after similar')
    similars.append(theScore)

    all_results.append({
        "test_case": tc["name"],
        "filters": tc["filters"],
        "summary": summary,
        "generation_time": gen_time,
        "evaluation": eval_result,
    })

    # Rate limit buffer for Gemini free tier
    time.sleep(5)

print(f"\n\n✅ All {len(TEST_CASES)} test cases completed!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



TEST CASE 1/9: Italian Restaurants (General)
Retrieved 10257 chars of review data
Generating summary...
Summary generated in 10.1s

--- Generated Summary ---
Overall Sentiment  
Positive — The reviews highlight positive experiences across multiple businesses, including happy hour options, satisfying meals, and exceptional service.

Top Praised Themes  
- **Brio Italian Grille**: Complimentary bread and chips, bruschetta pasta, and salads are praised.  
- **LaScala’s**: Gnocchis with marinara, baked eggplant, and a good service.  
- **Anthony’s at Paxon Hollow**: Excellent food, quick service, and a welcoming atmosphere.  

Top Complaints  
- **Caleco's Bar & Grill**: Cold artichoke dip, stale chips, and a poor pricing decision.  
- **Limoncello Ristorante & Caterers**: Inconsistent food quality and unappealing menu choices.  
- **Spasso Italian Grill**: Negative experience due to poor quality and lack of a children-friendly option.  

Representative Quotes  
1. "[exact quote from a re

## 6️⃣ Results Summary Table

In [ ]:
import pandas as pd

#case 1:
#case 2:
#CASE 3:
#case 4:
#case 5:
#case 6:
#case 7:
#case 8:
#case 9:
#every case study had an extremely high doc2vec similarity except for the fake business. There, no vocabulary was established since the model was not working with any legitmate reviews. However, the other results suggest the model accurately reflects semantics present in reviews.
rows = []
for r in all_results:
    auto = r["evaluation"].get("auto", {})
    judge = r["evaluation"].get("judge", {})

    row = {
        "Test Case": r["test_case"],
        "Gen Time (s)": round(r["generation_time"], 1),
        "Auto Score": auto.get("overall_auto_score", "N/A"),
        "Format": auto.get("format_compliance", {}).get("score", "N/A"),
        "Quotes": auto.get("quote_faithfulness", {}).get("score", "N/A"),
        "Coverage": auto.get("theme_coverage", {}).get("score", "N/A"),
        "No Halluc.": auto.get("hallucination_check", {}).get("score", "N/A"),
    }

    # Add judge scores if available
    if not judge.get("skipped") and not judge.get("error"):
        row["Judge Score"] = judge.get("overall_judge_score", "N/A")
        for dim in ["faithfulness", "completeness", "coherence", "relevance", "quote_accuracy"]:
            if dim in judge:
                row[f"J-{dim[:5].title()}"] = judge[dim]["score"]
    else:
        row["Judge Score"] = "N/A"

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df['doc2vec similarity']=similars
print("\n📊 EVALUATION RESULTS SUMMARY")
print("=" * 100)
display(results_df)

# Calculate averages (excluding N/A)
numeric_cols = results_df.select_dtypes(include='number').columns
print("\n📈 AVERAGES:")
for col in numeric_cols:
    valid = results_df[col].dropna()
    if len(valid) > 0:
        print(f"   {col}: {valid.mean():.2f}")


📊 EVALUATION RESULTS SUMMARY


,Test Case,Gen Time (s),Auto Score,Format,Quotes,Coverage,No Halluc.,Judge Score
0,Italian Restaurants (General),11.4,0.51,1.0,0.0,0.23,1.00,N/A
1,Mexican Restaurants,8.8,0.48,1.0,0.0,0.64,0.50,N/A
2,Coffee Shops,9.3,0.31,1.0,0.0,0.45,0.00,N/A
3,1-Star Reviews Only,9.4,0.53,1.0,0.5,0.71,0.00,N/A
4,5-Star Reviews Only,8.8,0.45,1.0,0.0,0.25,0.75,N/A
5,No Results (Fake Business),5.5,0.41,0.8,0.0,0.00,1.00,N/A
6,phrase query 1,10.1,0.44,1.0,0.5,0.38,0.00,N/A
7,phrase query 2,6.9,0.52,1.0,0.0,0.29,1.00,N/A
8,phrase query 3,9.4,0.28,1.0,0.0,0.30,0.00,N/A



📈 AVERAGES:
   Gen Time (s): 8.84
   Auto Score: 0.44
   Format: 0.98
   Quotes: 0.11
   Coverage: 0.36
   No Halluc.: 0.47


## 7️⃣ Model Comparison (0.6B vs 1.7B vs 4B)

Compare summarization quality across different Qwen3 model sizes.

> ⚠️ This section loads multiple models sequentially. Each model download may take a few minutes the first time.

In [ ]:
import gc

MODELS_TO_COMPARE = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B",
    "Qwen/Qwen3-4B",
]

# Use one consistent test case for fair comparison
COMPARISON_FILTERS = {"categories": "Italian", "k": 80}

addDict={"business_name": None,
                "city": None,
                "state": None,
                "review_stars": None,
                "categories": "Italian"}
print("Retrieving reviews for comparison test...")
baseline="good customer service and large wine selection"
comparison_docs = rag_retrival.retrieve_reviews_for_summary(
    vs, addDict,baseline,20,False
)
print(f"Retrieved {len(comparison_docs)} chars of review data")

model_results = []
similars=[]
for model_name in MODELS_TO_COMPARE:
    print(f"\n{'='*60}")
    print(f"🤖 Loading {model_name}...")
    print(f"{'='*60}")

    try:
        tok_cmp, model_cmp = load_model(model_name)

        print("Generating summary...")
        start_time = time.time()
        summary = summarize_reviews(comparison_docs, tok_cmp, model_cmp)
        gen_time = time.time() - start_time
        print(f"Generated in {gen_time:.1f}s")
        print(f"\n--- Summary ---\n{summary}\n--- End ---\n")

        # Evaluate
        eval_result = run_full_evaluation(
            summary, comparison_docs,
            api_key=GEMINI_API_KEY,
            gemini_model=GEMINI_MODEL
        )
        
        model_results.append({
            "model": model_name,
            "summary": summary,
            "gen_time": gen_time,
            "evaluation": eval_result,
        })
        theScore=similarEval(summary,comparison_docs)
        print(theScore)
        print('got here after similar')
        similars.append(theScore)

        # Free GPU memory before loading next model
        del model_cmp, tok_cmp
        gc.collect()
        torch.cuda.empty_cache()

        time.sleep(5)  # Rate limit buffer

    except Exception as e:
        print(f"❌ Failed to load {model_name}: {e}")
        model_results.append({
            "model": model_name,
            "error": str(e),
        })

print(f"\n✅ Model comparison complete!")

Retrieving reviews for comparison test...
Retrieved 10257 chars of review data

🤖 Loading Qwen/Qwen3-0.6B...
Loading model: Qwen/Qwen3-0.6B ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on CUDA (dtype=torch.bfloat16)
Generating summary...
Generated in 13.4s

--- Summary ---
## Overall Sentiment  
Positive — The reviews highlight positive experiences across multiple businesses, including exceptional food, welcoming ambiance, and customer satisfaction.

## Top Praised Themes  
1. **Exceptional Food**: All reviewed restaurants feature top-rated meals such as venison Milanese, grilled cheese sandwiches, and artisanal desserts.  
2. **Widespread Community Atmosphere**: Many reviews mention lively settings and a strong sense of community, making them enjoyable for diners.  
3. **Friendly Service**: Positive feedback includes praise for attentive staff and efficient operations, contributing to overall satisfaction.

## Top Complaints  
1. **Slow Service**: Some reviews note delays in processing orders and response times, particularly affecting customers who need immediate assistance.  
2. **Overwhelming Menu Options**: A few comments express frustration with the


📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.3/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.0/1.00
   Theme Coverage:       0.38/1.00
   Hallucination Check:  0/1.00
   ⚠️  Unverified quotes: ['[Brio Italian Grille] my friends at work connoisseurs of good food in the St Louis area introduced me to yet another very nice restaurant...', "[Anthony's at Paxon Hollow] our meal was excellent...we both appreciated the only problem was trying to pick something from the menu..."]
   ⚠️  Flags: ["Possible hallucinated business name: 'Exceptional Food'", "Possible hallucinated business name: 'Widespread Community Atmosphere'", "Possible hallucinated business name: 'Friendly Service'"]

❌ LLM Judge Error: Gemini API error: 403 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your API key was reported as leaked. Please use another API key.


🤖 Loading Qwen/Qwen3-1.7B...
Loading model: Qwen

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded on CUDA (dtype=torch.bfloat16)
Generating summary...
Generated in 23.0s

--- Summary ---
## Overall Sentiment
Positive — Most reviews highlight positive experiences with food, service, and ambiance, though some mention issues that affected overall satisfaction.

## Top Praised Themes
- **Food Quality**: "The food come out very quickly and it is phenomenal" (Portobello Cafe), "everything tasted fresh and homemade" (Anthony's at Paxon Hollow).
- **Service**: "the waiters there funny and nice" (Portobello Cafe), "our server was good" (Caleco's Bar & Grill), "staff are as friendly as it gets" (Romano's Macaroni Grill).
- **Ambiance**: "it feels like your are entering someones home" (Portobello Cafe), "beautiful place the service adds to the charm" (Brio Italian Grille).

## Top Complaints
- **Inconsistent Food Quality**: "sometimes this food is very very good unfortunately its not consistent" (Limoncello Ristorante & Caterers), "my tomatobasil soup tasted kind of like a weak m


📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.53/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.71/1.00
   Theme Coverage:       0.46/1.00
   Hallucination Check:  0/1.00
   ⚠️  Unverified quotes: ["we didn't see a children's menu", 's at Paxon Hollow).\n- **Service**: "the waiters there funny and nice" (Portobello Cafe), "our server was good" (Caleco']
   ⚠️  Flags: ["Possible hallucinated business name: 'Food Quality'", "Possible hallucinated business name: 'Inconsistent Food Quality'", "Possible hallucinated business name: 'Slow Service'"]

❌ LLM Judge Error: Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exc

In [ ]:
# Build comparison table
comp_rows = []
for r in model_results:
    if "error" in r:
        comp_rows.append({"Model": r["model"], "Error": r["error"]})
        continue

    auto = r["evaluation"].get("auto", {})
    judge = r["evaluation"].get("judge", {})

    row = {
        "Model": r["model"].split("/")[-1],
        "Gen Time (s)": round(r["gen_time"], 1),
        "Auto Score": auto.get("overall_auto_score", "N/A"),
        "Format": auto.get("format_compliance", {}).get("score", "N/A"),
        "Quotes": auto.get("quote_faithfulness", {}).get("score", "N/A"),
        "Coverage": auto.get("theme_coverage", {}).get("score", "N/A"),
        "No Halluc.": auto.get("hallucination_check", {}).get("score", "N/A"),
    }

    if not judge.get("skipped") and not judge.get("error"):
        row["Judge Score"] = judge.get("overall_judge_score", "N/A")
        for dim in ["faithfulness", "completeness", "coherence", "relevance", "quote_accuracy"]:
            if dim in judge:
                row[f"J-{dim[:5].title()}"] = judge[dim]["score"]

    comp_rows.append(row)

comp_df = pd.DataFrame(comp_rows)
comp_df['doc2vec similarity']=similars
print("\n📊 MODEL COMPARISON RESULTS")
print("=" * 100)
print(f"Test case: {COMPARISON_FILTERS}")
display(comp_df)


📊 MODEL COMPARISON RESULTS
Test case: {'categories': 'Italian', 'k': 80}


,Model,Gen Time (s),Auto Score,Format,Quotes,Coverage,No Halluc.
0,Qwen3-0.6B,13.4,0.30,1.0,0.00,0.38,0
1,Qwen3-1.7B,15.7,0.44,1.0,0.29,0.62,0
2,Qwen3-4B,23.0,0.53,1.0,0.71,0.46,0


## 8️⃣ Save Results

In [ ]:
import json

# Save results to Google Drive
save_dir = "/content/drive/MyDrive/NLP-Project/eval_results"
os.makedirs(save_dir, exist_ok=True)

# Save test case results
results_df.to_csv(f"{save_dir}/test_case_results.csv", index=False)

# Save model comparison results
comp_df.to_csv(f"{save_dir}/model_comparison_results.csv", index=False)

# Save detailed results as JSON
serializable_results = []
for r in all_results:
    sr = {k: v for k, v in r.items() if k != 'evaluation'}
    sr['evaluation'] = r['evaluation']
    serializable_results.append(sr)

with open(f"{save_dir}/detailed_results.json", "w") as f:
    json.dump(serializable_results, f, indent=2, default=str)

print(f"✅ Results saved to {save_dir}/")
print(f"   • test_case_results.csv")
print(f"   • model_comparison_results.csv")
print(f"   • detailed_results.json")